# Member 3: Lead Data Scientist

This notebook tracks the first explainable modeling workflow for Team Jarvis.

## Modeling Objective

Estimate each outlet's latent maximum monthly purchase potential in liters for January 2026.

## Latent Potential Idea

Historical sales may be censored by stockouts, credit limits, delivery caps, or other operational constraints.

`Observed Sales = min(True Demand Potential, Constraints)`

The first model estimates hidden demand potential using high-but-credible historical outlet months, not simple average sales.

## Data Inputs

- `data/silver/outlet_master.csv`
- `data/silver/transactions_history_final.csv`
- `data/silver/distributor_seasonality_details.csv`
- Optional later: `data/gold/master_features.csv`

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SILVER = PROJECT_ROOT / "data" / "silver"
required_files = [
    SILVER / "outlet_master.csv",
    SILVER / "transactions_history_final.csv",
    SILVER / "distributor_seasonality_details.csv",
]
all(path.exists() for path in required_files)

In [ ]:
if all(path.exists() for path in required_files):
    outlet_master = pd.read_csv(SILVER / "outlet_master.csv")
    transactions = pd.read_csv(SILVER / "transactions_history_final.csv")
    seasonality = pd.read_csv(SILVER / "distributor_seasonality_details.csv")
    print(outlet_master.shape, transactions.shape, seasonality.shape)
else:
    print("Silver files are not available yet in this working tree.")

## Baseline Modeling

The baseline uses outlet-level monthly history: recent average, p75, p90, max, active month count, transaction count, total liters, and average bill value.

In [ ]:
from src.models.baseline_potential import build_baseline_potential

if all(path.exists() for path in required_files):
    baseline = build_baseline_potential(transactions)
    baseline.head()
else:
    baseline = None

## Seasonality Adjustment

January 2026 seasonality is inferred from historical January distributor labels using simple multipliers: Favorable = 1.08, Moderate = 1.00, Un-Favorable = 0.92.

In [ ]:
from src.models.seasonality import apply_january_seasonality

if baseline is not None:
    seasonality_adjusted = apply_january_seasonality(baseline, transactions, seasonality)
    seasonality_adjusted[["Outlet_ID", "january_seasonality_multiplier", "seasonality_adjusted_potential_liters"]].head()
else:
    seasonality_adjusted = None

## Peer Benchmarking

Peer benchmarks compare outlets within Outlet_Type + Outlet_Size groups, falling back to Outlet_Type and then global benchmarks when groups are too small.

In [ ]:
from src.models.peer_benchmark import build_peer_benchmarks

if baseline is not None:
    peer_benchmark = build_peer_benchmarks(outlet_master, baseline)
    peer_benchmark.head()
else:
    peer_benchmark = None

## Final Blend

The first blend combines baseline, January seasonality, and peer benchmark estimates. If `data/gold/master_features.csv` is available later, numeric Gold signals are included with a very small conservative multiplier.

In [ ]:
from src.pipeline.make_submission import make_submission

# Run this from the repository root after Silver files are available:
# predictions = make_submission()

## Submission Validation

The validator checks schema, row count, outlet coverage, duplicates, missing values, and positive prediction values.

In [ ]:
from src.pipeline.validate_submission import validate_submission

# Run after creating submissions/teamname_predictions.csv:
# validate_submission()

## Notes for Final Report

- Historical sales are treated as censored observations.
- The baseline is intentionally explainable and conservative.
- Peer benchmarking reduces underestimation for outlets with suppressed history.
- Seasonality adjusts the January 2026 target month without creating a black-box model.
- Gold/POI features can be integrated later once Member 2 finalizes `data/gold/master_features.csv`.